# Busca em largura e ordenação topológica — Tutorial

**Algoritmos e Estruturas de Dados 2 — DCOMP/UFS**
Prof. Dr. André Yoshiaki Kashiwabara

Este tutorial acompanha a aula e segue de perto o material
[*Algoritmos para Grafos (em linguagem C)*](https://www.ime.usp.br/~pf/algoritmos_para_grafos/),
de Paulo Feofiloff (IME-USP). O código é escrito em C e compilado com `gcc` a partir
do próprio notebook: cada célula `%%writefile` grava um programa completo em `src/`,
e a célula seguinte compila e executa.

## Objetivos

Ao final deste tutorial você será capaz de:

- implementar uma **fila circular** sobre um vetor e usá-la para escrever a busca em
  largura sobre matriz de adjacências e sobre listas de adjacência;
- explicar por que `dist[]` produzido pela BFS é o **número mínimo de arcos** da raiz
  até cada vértice, e recuperar um caminho mínimo a partir de `pai[]`;
- reconhecer as **camadas** da busca e o que muda (e o que não muda) quando a ordem
  de enumeração dos vizinhos muda;
- ordenar topologicamente um DAG por **Kahn** (graus de entrada + fila) e por
  **pós-ordem invertida da DFS**, e detectar ciclos com o mesmo código;
- aplicar ordenação topológica a um problema de escalonamento.

O grafo condutor é o mesmo das aulas anteriores:

```
0-1   0-5   1-0   1-5   2-4   3-1   5-3
```

In [ ]:
# Preparação do ambiente: cria a pasta de trabalho e confere o compilador.
import os, subprocess

os.makedirs('src', exist_ok=True)
print(subprocess.run(['gcc', '--version'], capture_output=True, text=True).stdout.splitlines()[0])

def compilar_e_rodar(fonte, entrada=None):
    """Compila src/<fonte> com gcc e executa, mostrando a saída."""
    exe = os.path.join('src', os.path.splitext(fonte)[0])
    comp = subprocess.run(['gcc', '-std=c99', '-Wall', '-o', exe, os.path.join('src', fonte)],
                          capture_output=True, text=True)
    if comp.returncode != 0:
        print('ERRO DE COMPILACAO:\n' + comp.stderr)
        return
    if comp.stderr.strip():
        print('avisos do compilador:\n' + comp.stderr.strip() + '\n' + '-' * 40)
    run = subprocess.run([exe], input=entrada, capture_output=True, text=True)
    print(run.stdout, end='')
    if run.stderr.strip():
        print('stderr:\n' + run.stderr)

## 1. A fila circular e a busca em largura

A busca em largura é a busca em profundidade com uma troca: a pilha dá lugar a uma
**fila**. Retiramos da fila o vértice descoberto há mais tempo, examinamos seus
vizinhos e enfileiramos os que ainda não foram descobertos.

Dois detalhes fazem o algoritmo funcionar:

1. o vértice é marcado **no momento em que entra na fila** (não quando sai) — assim
   nenhum vértice entra duas vezes e a fila nunca passa de `V` elementos;
2. quem entra recebe `dist[v] + 1`, onde `v` é o vértice que acabou de sair. Isso
   mantém a fila sempre com, no máximo, duas camadas ao mesmo tempo — e é a razão
   de `dist[]` sair mínima.

A fila é um vetor de `V` posições usado de forma **circular**: os índices `ini` e
`fim` avançam módulo `cap`, e `qtd` diz quantos elementos há.

**Fonte:** Feofiloff (2020), cap. *Busca em largura* (código em C e a fila circular); Cormen et al. (2009), §22.2 (pseudocódigo `BFS`, lema 22.3).

In [ ]:
%%writefile src/bfs_matriz.c

/* BUSCA EM LARGURA - representacao por matriz de adjacencias.
   Adaptado de P. Feofiloff, "Algoritmos para Grafos (em linguagem C)", IME-USP.
   https://www.ime.usp.br/~pf/algoritmos_para_grafos/ */
#include <stdio.h>
#include <stdlib.h>

#define vertex int
#define maxV 100

struct graph { int V; int A; int **adj; };
typedef struct graph *Graph;

static int **MATRIXint( int r, int c, int val) {
   int **m = malloc( r * sizeof (int *));
   for (int i = 0; i < r; ++i)
      m[i] = malloc( c * sizeof (int));
   for (int i = 0; i < r; ++i)
      for (int j = 0; j < c; ++j)
         m[i][j] = val;
   return m;
}

Graph GRAPHinit( int V) {
   Graph G = malloc( sizeof *G);
   G->V = V;  G->A = 0;
   G->adj = MATRIXint( V, V, 0);
   return G;
}

void GRAPHinsertArc( Graph G, vertex v, vertex w) {
   if (G->adj[v][w] == 0) { G->adj[v][w] = 1; ++G->A; }
}

/* ---------- fila circular sobre um vetor de cap posicoes ---------- */
static vertex *fila;
static int cap, ini, fim, qtd;

void QUEUEinit( int N)      { cap = N;  fila = malloc( cap * sizeof (vertex));
                              ini = fim = qtd = 0; }
int  QUEUEempty( void)      { return qtd == 0; }
void QUEUEput( vertex v)    { fila[fim] = v;  fim = (fim + 1) % cap;  ++qtd; }
vertex QUEUEget( void)      { vertex v = fila[ini];  ini = (ini + 1) % cap;
                              --qtd;  return v; }
void QUEUEfree( void)       { free( fila); }

/* ---------- busca em largura ---------- */
static int dist[maxV], pai[maxV];

void GRAPHbfs( Graph G, vertex s) {
   for (vertex v = 0; v < G->V; ++v) dist[v] = pai[v] = -1;
   QUEUEinit( G->V);
   dist[s] = 0;                      /* marca s ao ENTRAR na fila */
   QUEUEput( s);
   printf( "ordem de visita:");
   while (!QUEUEempty()) {
      vertex v = QUEUEget();         /* o mais antigo da fila */
      printf( " %d", v);
      for (vertex w = 0; w < G->V; ++w)          /* vizinhos de v */
         if (G->adj[v][w] != 0 && dist[w] == -1) {
            dist[w] = dist[v] + 1;
            pai[w]  = v;
            QUEUEput( w);
         }
   }
   printf( "\n");
   QUEUEfree();
}

/* imprime o caminho minimo de s a t, lendo pai[] de t para tras */
void GRAPHpath( vertex s, vertex t) {
   if (dist[t] == -1) { printf( "nao ha caminho de %d a %d\n", s, t); return; }
   vertex pilha[maxV];  int k = 0;
   for (vertex v = t; v != -1; v = pai[v]) pilha[k++] = v;
   printf( "caminho minimo de %d a %d (%d arcos):", s, t, dist[t]);
   while (k > 0) printf( " %d", pilha[--k]);
   printf( "\n");
}

int main( void) {
   Graph G = GRAPHinit( 6);
   int arcos[7][2] = {{0,1},{0,5},{1,0},{1,5},{2,4},{3,1},{5,3}};
   for (int i = 0; i < 7; ++i) GRAPHinsertArc( G, arcos[i][0], arcos[i][1]);

   GRAPHbfs( G, 0);
   printf( "vertice: ");  for (vertex v = 0; v < G->V; ++v) printf( "%3d", v);
   printf( "\ndist[]:  ");  for (vertex v = 0; v < G->V; ++v) printf( "%3d", dist[v]);
   printf( "\npai[]:   ");  for (vertex v = 0; v < G->V; ++v) printf( "%3d", pai[v]);
   printf( "\n");
   GRAPHpath( 0, 3);
   GRAPHpath( 0, 4);
   return 0;
}

In [ ]:
compilar_e_rodar('bfs_matriz.c')
# Saida esperada: ordem de visita 0 1 5 3; dist[] = 0 1 -1 2 -1 1;
# pai[] = -1 0 -1 5 -1 0; caminho minimo de 0 a 3 = 0 5 3 (2 arcos);
# os vertices 2 e 4 nao sao alcancaveis a partir de 0.

## 2. A mesma busca sobre listas de adjacência

O corpo do algoritmo não muda: muda **como** enumeramos os vizinhos. Em vez de
varrer a linha `v` da matriz (sempre `V` testes), percorremos `G->adj[v]` (tantos
passos quanto o grau de saída de `v`).

Como `GRAPHinsertArc()` insere cada nó **no início** da lista, os vizinhos saem na
ordem inversa à de inserção. Repare no resultado: a **ordem de visita muda** de
`0 1 5 3` para `0 5 1 3`, mas `dist[]` e `pai[]` ficam **idênticos**. A ordem dentro
de uma camada é arbitrária; as distâncias, não.

**Fonte:** Feofiloff (2020), caps. *Estruturas de dados para grafos* e *Busca em largura*; Sedgewick (2002), *Algorithms in C, Part 5*, cap. 18 (implementação em C).

In [ ]:
%%writefile src/bfs_listas.c

/* BUSCA EM LARGURA - representacao por listas de adjacencia. */
#include <stdio.h>
#include <stdlib.h>

#define vertex int
#define maxV 100

typedef struct node *link;
struct node { vertex w; link next; };
struct graph { int V; int A; link *adj; };
typedef struct graph *Graph;

static link NEWnode( vertex w, link next) {
   link a = malloc( sizeof (struct node));
   a->w = w;  a->next = next;
   return a;
}

Graph GRAPHinit( int V) {
   Graph G = malloc( sizeof *G);
   G->V = V;  G->A = 0;
   G->adj = malloc( V * sizeof (link));
   for (vertex v = 0; v < V; ++v) G->adj[v] = NULL;
   return G;
}

/* insere o arco v-w NO INICIO da lista de v */
void GRAPHinsertArc( Graph G, vertex v, vertex w) {
   for (link a = G->adj[v]; a != NULL; a = a->next)
      if (a->w == w) return;
   G->adj[v] = NEWnode( w, G->adj[v]);
   ++G->A;
}

static vertex *fila;
static int cap, ini, fim, qtd;
void QUEUEinit( int N)   { cap = N;  fila = malloc( cap * sizeof (vertex));
                           ini = fim = qtd = 0; }
int  QUEUEempty( void)   { return qtd == 0; }
void QUEUEput( vertex v) { fila[fim] = v;  fim = (fim + 1) % cap;  ++qtd; }
vertex QUEUEget( void)   { vertex v = fila[ini];  ini = (ini + 1) % cap;
                           --qtd;  return v; }

static int dist[maxV], pai[maxV];

void GRAPHbfs( Graph G, vertex s) {
   for (vertex v = 0; v < G->V; ++v) dist[v] = pai[v] = -1;
   QUEUEinit( G->V);
   dist[s] = 0;
   QUEUEput( s);
   printf( "ordem de visita:");
   while (!QUEUEempty()) {
      vertex v = QUEUEget();
      printf( " %d", v);
      for (link a = G->adj[v]; a != NULL; a = a->next)
         if (dist[a->w] == -1) {
            dist[a->w] = dist[v] + 1;
            pai[a->w]  = v;
            QUEUEput( a->w);
         }
   }
   printf( "\n");
   free( fila);
}

int main( void) {
   Graph G = GRAPHinit( 6);
   int arcos[7][2] = {{0,1},{0,5},{1,0},{1,5},{2,4},{3,1},{5,3}};
   for (int i = 0; i < 7; ++i) GRAPHinsertArc( G, arcos[i][0], arcos[i][1]);

   printf( "listas de adjacencia (ordem inversa a de insercao):\n");
   for (vertex v = 0; v < G->V; ++v) {
      printf( "  adj[%d]:", v);
      for (link a = G->adj[v]; a != NULL; a = a->next) printf( " %d", a->w);
      printf( "\n");
   }
   GRAPHbfs( G, 0);
   printf( "dist[]:  ");  for (vertex v = 0; v < G->V; ++v) printf( "%3d", dist[v]);
   printf( "\npai[]:   ");  for (vertex v = 0; v < G->V; ++v) printf( "%3d", pai[v]);
   printf( "\n");
   return 0;
}

In [ ]:
compilar_e_rodar('bfs_listas.c')
# Compare com o exemplo anterior: ordem de visita 0 5 1 3 (mudou),
# dist[] e pai[] inalterados.

### Exercício 1 — caminho mínimo de $s$ a $t$

Escreva `GRAPHshortestPath(G, s, t)`, que imprime um caminho com o **menor número de
arcos** de `s` a `t`, ou avisa que não existe caminho.

A BFS já está pronta no arquivo: ela preenche `dist[]` e `pai[]`. Falta ler `pai[]`
de `t` para trás — o que produz o caminho ao contrário — e imprimi-lo na ordem
correta. Um vetor usado como pilha resolve.

Atenção ao caso `dist[t] == -1`: não há caminho.

**Fonte:** Cormen et al. (2009), §22.2 — lema 22.6 (o subgrafo de predecessores é uma árvore de caminhos mínimos) e teorema 22.7 (`PRINT-PATH`).

In [ ]:
%%writefile src/ex1_caminho.c

#include <stdio.h>
#include <stdlib.h>

#define vertex int
#define maxV 100

typedef struct node *link;
struct node { vertex w; link next; };
struct graph { int V; int A; link *adj; };
typedef struct graph *Graph;

static link NEWnode( vertex w, link next) {
   link a = malloc( sizeof (struct node));  a->w = w;  a->next = next;  return a;
}
Graph GRAPHinit( int V) {
   Graph G = malloc( sizeof *G);  G->V = V;  G->A = 0;
   G->adj = malloc( V * sizeof (link));
   for (vertex v = 0; v < V; ++v) G->adj[v] = NULL;
   return G;
}
void GRAPHinsertArc( Graph G, vertex v, vertex w) {
   G->adj[v] = NEWnode( w, G->adj[v]);  ++G->A;
}

static vertex *fila;
static int cap, ini, fim, qtd;
void QUEUEinit( int N)   { cap = N;  fila = malloc( cap * sizeof (vertex));
                           ini = fim = qtd = 0; }
int  QUEUEempty( void)   { return qtd == 0; }
void QUEUEput( vertex v) { fila[fim] = v;  fim = (fim + 1) % cap;  ++qtd; }
vertex QUEUEget( void)   { vertex v = fila[ini];  ini = (ini + 1) % cap;
                           --qtd;  return v; }

static int dist[maxV], pai[maxV];

/* BFS pronta: preenche dist[] e pai[] a partir de s. */
void GRAPHbfs( Graph G, vertex s) {
   for (vertex v = 0; v < G->V; ++v) dist[v] = pai[v] = -1;
   QUEUEinit( G->V);
   dist[s] = 0;  QUEUEput( s);
   while (!QUEUEempty()) {
      vertex v = QUEUEget();
      for (link a = G->adj[v]; a != NULL; a = a->next)
         if (dist[a->w] == -1) {
            dist[a->w] = dist[v] + 1;  pai[a->w] = v;
            QUEUEput( a->w);
         }
   }
}

/* EXERCICIO: imprima um caminho MINIMO de s a t, no formato
      0 5 3
   ou a mensagem "nao ha caminho de 0 a 4".
   Sugestao: rode GRAPHbfs(G, s) e depois siga pai[] de t para tras,
   guardando os vertices em uma pilha (um vetor e um contador bastam). */
void GRAPHshortestPath( Graph G, vertex s, vertex t) {
   GRAPHbfs( G, s);
   /* TODO: implemente aqui */
}

int main( void) {
   Graph G = GRAPHinit( 6);
   int arcos[7][2] = {{0,1},{0,5},{1,0},{1,5},{2,4},{3,1},{5,3}};
   for (int i = 0; i < 7; ++i) GRAPHinsertArc( G, arcos[i][0], arcos[i][1]);

   printf( "0 -> 3  esperado: 0 5 3\n        obtido:   ");
   GRAPHshortestPath( G, 0, 3);   puts( "---");

   printf( "0 -> 1  esperado: 0 1\n        obtido:   ");
   GRAPHshortestPath( G, 0, 1);   puts( "---");

   printf( "0 -> 4  esperado: nao ha caminho de 0 a 4\n        obtido:   ");
   GRAPHshortestPath( G, 0, 4);   puts( "---");
   return 0;
}

In [ ]:
compilar_e_rodar('ex1_caminho.c')

## 3. As camadas, vistas de perto

`dist[]` particiona os vértices alcançáveis em **camadas**: a camada $k$ é o conjunto
dos vértices a exatamente $k$ arcos da raiz. A propriedade que sustenta a corretude é
que **todo arco do grafo liga vértices da mesma camada ou de camadas consecutivas** —
nenhum arco salta uma camada à frente.

A célula abaixo refaz a busca em Python puro, apenas para enxergar as camadas e o
caminho reconstruído por `pai[]`. É o mesmo algoritmo do código em C, com
`collections.deque` no lugar da fila circular.

**Fonte:** Cormen et al. (2009), §22.2 — lemas 22.1 a 22.3, corolário 22.4 e teorema 22.5 (corretude de `dist[]`); Sedgewick e Wayne (2011), §4.1.

In [ ]:
from collections import deque

# grafo da aula, em listas de adjacencia ordenadas (vizinhos em ordem crescente)
G = {0: [1, 5], 1: [0, 5], 2: [4], 3: [1], 4: [], 5: [3]}

def bfs(G, s):
    """Devolve (dist, pai) pela busca em largura a partir de s."""
    dist = {v: -1 for v in G}
    pai = {v: None for v in G}
    dist[s] = 0
    fila = deque([s])
    ordem = []
    while fila:
        v = fila.popleft()
        ordem.append(v)
        for w in G[v]:
            if dist[w] == -1:
                dist[w] = dist[v] + 1
                pai[w] = v
                fila.append(w)
    return ordem, dist, pai

ordem, dist, pai = bfs(G, 0)
print("ordem de visita:", ordem)

k = 0
while any(d == k for d in dist.values()):
    print(f"camada {k}: {[v for v in G if dist[v] == k]}")
    k += 1
print("nao alcancaveis:", [v for v in G if dist[v] == -1])

def caminho(pai, t):
    p = []
    while t is not None:
        p.append(t)
        t = pai[t]
    return list(reversed(p))

print("caminho minimo 0 -> 3:", caminho(pai, 3), f"({dist[3]} arcos)")

## 4. Ordenação topológica: o algoritmo de Kahn

Mudamos de problema. Dado um grafo de **dependências** ($v \to w$ significa "$v$
precisa estar pronto antes de $w$"), queremos uma ordem de escrever os módulos de um
projeto que respeite todos os arcos. Isso é uma **ordenação topológica**, e ela existe
se e somente se o grafo é acíclico (um **DAG**).

O algoritmo de Kahn é a BFS com uma condição de entrada mais forte: o vértice entra na
fila quando **todos** os seus predecessores já saíram, não ao ser visto pela primeira
vez. Quem controla isso é o vetor `indeg[]` de graus de entrada — decrementar
`indeg[w]` é dizer "mais uma dependência de $w$ ficou pronta".

O DAG usado aqui é:

| $v$ | módulo | depende de |
|---|---|---|
| 0 | `grafo.h` — a API do grafo | — |
| 1 | `grafo.c` — lista de adjacência | 0 |
| 2 | `fila.c` — fila circular | — |
| 3 | `bfs.c` — busca em largura | 1, 2 |
| 4 | `topo.c` — Kahn | 3 |
| 5 | `main.c` — liga tudo e testa | 2, 4 |

Note o teste final `return k == G->V`: se a fila esvaziou antes de emitir todos os
vértices, os que sobraram estão presos em um ciclo. **O mesmo código responde às duas
perguntas** — "qual é uma ordem válida?" e "existe dependência circular?".

**Fonte:** Kahn (1962), *Communications of the ACM* 5(11):558–562 (algoritmo original); Levitin (2012), §4.2 (o mesmo método, por remoção de fontes); Feofiloff (2020), cap. *Ordenação topológica*.

In [ ]:
%%writefile src/kahn.c

/* ORDENACAO TOPOLOGICA - algoritmo de Kahn (graus de entrada + fila). */
#include <stdio.h>
#include <stdlib.h>

#define vertex int
#define maxV 100

typedef struct node *link;
struct node { vertex w; link next; };
struct graph { int V; int A; link *adj; };
typedef struct graph *Graph;

static const char *nome[] = { "grafo.h", "grafo.c", "fila.c",
   "bfs.c", "topo.c", "main.c" };

static link NEWnode( vertex w, link next) {
   link a = malloc( sizeof (struct node));  a->w = w;  a->next = next;  return a;
}
Graph GRAPHinit( int V) {
   Graph G = malloc( sizeof *G);  G->V = V;  G->A = 0;
   G->adj = malloc( V * sizeof (link));
   for (vertex v = 0; v < V; ++v) G->adj[v] = NULL;
   return G;
}
void GRAPHinsertArc( Graph G, vertex v, vertex w) {
   G->adj[v] = NEWnode( w, G->adj[v]);  ++G->A;
}

static vertex *fila;
static int cap, ini, fim, qtd;
void QUEUEinit( int N)   { cap = N;  fila = malloc( cap * sizeof (vertex));
                           ini = fim = qtd = 0; }
int  QUEUEempty( void)   { return qtd == 0; }
void QUEUEput( vertex v) { fila[fim] = v;  fim = (fim + 1) % cap;  ++qtd; }
vertex QUEUEget( void)   { vertex v = fila[ini];  ini = (ini + 1) % cap;
                           --qtd;  return v; }

/* Escreve em ord[] uma ordenacao topologica de G e devolve 1;
   se G tem ciclo, devolve 0 (e ord[] fica com os k < V vertices emitidos). */
int GRAPHtopoKahn( Graph G, vertex ord[]) {
   int indeg[maxV], k = 0;
   for (vertex v = 0; v < G->V; ++v) indeg[v] = 0;
   for (vertex v = 0; v < G->V; ++v)                /* graus de entrada */
      for (link a = G->adj[v]; a != NULL; a = a->next)
         ++indeg[a->w];
   QUEUEinit( G->V);
   for (vertex v = 0; v < G->V; ++v)
      if (indeg[v] == 0) QUEUEput( v);              /* nao dependem de ninguem */
   while (!QUEUEempty()) {
      vertex v = QUEUEget();
      ord[k++] = v;                                 /* v entra na saida */
      for (link a = G->adj[v]; a != NULL; a = a->next)
         if (--indeg[a->w] == 0) QUEUEput( a->w);
   }
   free( fila);
   return k == G->V;                                 /* k < V  =>  ha ciclo */
}

int main( void) {
   /* DAG de dependencias: v -> w significa "v fica pronto antes de w" */
   Graph G = GRAPHinit( 6);
   int pre[6][2] = {{0,1},{1,3},{2,3},{3,4},{4,5},{2,5}};
   for (int i = 0; i < 6; ++i) GRAPHinsertArc( G, pre[i][0], pre[i][1]);

   vertex ord[maxV];
   if (GRAPHtopoKahn( G, ord)) {
      printf( "e um DAG. Ordenacao topologica:\n");
      for (int k = 0; k < G->V; ++k)
         printf( "  %d. %d (%s)\n", k + 1, ord[k], nome[ord[k]]);
   } else
      printf( "ha ciclo: nao existe ordenacao topologica\n");

   /* agora com o arco 4->1 (grafo.c passa a depender de topo.c) */
   printf( "\nacrescentando o arco 4->1:\n");
   GRAPHinsertArc( G, 4, 1);
   if (!GRAPHtopoKahn( G, ord))
      printf( "  ha ciclo: nao existe ordenacao topologica\n");
   return 0;
}

In [ ]:
compilar_e_rodar('kahn.c')
# Saida esperada: ordenacao 0, 2, 1, 3, 4, 5 (grafo.h, fila.c, grafo.c,
# bfs.c, topo.c, main.c); depois do arco 4->1, ha ciclo.

### Exercício 2 — quem está no ciclo?

Quando Kahn falha, saber *que existe* um ciclo raramente basta: queremos saber **quais**
módulos estão envolvidos, para quebrar a dependência circular.

Complete `GRAPHtopoCiclo()` marcando `ciclo[v] = 1` para todo vértice que não foi
emitido. A dica está no próprio algoritmo: ao final do laço, um vértice ficou de fora
exatamente quando `indeg[v] > 0` — seu contador nunca zerou.

No grafo do teste, o arco `4 -> 1` cria o ciclo $1 \to 3 \to 4 \to 1$, e a resposta
esperada é `1 3 4`.

**Fonte:** Kahn (1962), pp. 558–562 (a contagem da saída como teste de aciclicidade); Cormen et al. (2009), §22.4, lema 22.11 (ciclo ⟺ arco de retorno na DFS).

In [ ]:
%%writefile src/ex2_ciclo.c

#include <stdio.h>
#include <stdlib.h>

#define vertex int
#define maxV 100

typedef struct node *link;
struct node { vertex w; link next; };
struct graph { int V; int A; link *adj; };
typedef struct graph *Graph;

static link NEWnode( vertex w, link next) {
   link a = malloc( sizeof (struct node));  a->w = w;  a->next = next;  return a;
}
Graph GRAPHinit( int V) {
   Graph G = malloc( sizeof *G);  G->V = V;  G->A = 0;
   G->adj = malloc( V * sizeof (link));
   for (vertex v = 0; v < V; ++v) G->adj[v] = NULL;
   return G;
}
void GRAPHinsertArc( Graph G, vertex v, vertex w) {
   G->adj[v] = NEWnode( w, G->adj[v]);  ++G->A;
}

static vertex *fila;
static int cap, ini, fim, qtd;
void QUEUEinit( int N)   { cap = N;  fila = malloc( cap * sizeof (vertex));
                           ini = fim = qtd = 0; }
int  QUEUEempty( void)   { return qtd == 0; }
void QUEUEput( vertex v) { fila[fim] = v;  fim = (fim + 1) % cap;  ++qtd; }
vertex QUEUEget( void)   { vertex v = fila[ini];  ini = (ini + 1) % cap;
                           --qtd;  return v; }

/* EXERCICIO: alem de escrever a ordenacao em ord[], preencha ciclo[] com 1
   nas posicoes dos vertices que NAO foram emitidos (esses sao exatamente os
   que estao em algum ciclo ou dependem de um). Devolva o numero k de
   vertices emitidos.
   Sugestao: ao final do laco, todo v com indeg[v] > 0 ficou de fora. */
int GRAPHtopoCiclo( Graph G, vertex ord[], int ciclo[]) {
   int indeg[maxV], k = 0;
   for (vertex v = 0; v < G->V; ++v) { indeg[v] = 0;  ciclo[v] = 0; }
   for (vertex v = 0; v < G->V; ++v)
      for (link a = G->adj[v]; a != NULL; a = a->next) ++indeg[a->w];
   QUEUEinit( G->V);
   for (vertex v = 0; v < G->V; ++v)
      if (indeg[v] == 0) QUEUEput( v);
   while (!QUEUEempty()) {
      vertex v = QUEUEget();
      ord[k++] = v;
      for (link a = G->adj[v]; a != NULL; a = a->next)
         if (--indeg[a->w] == 0) QUEUEput( a->w);
   }
   /* TODO: marque ciclo[v] = 1 para todo v que ficou de fora */
   return k;
}

int main( void) {
   Graph G = GRAPHinit( 6);
   int arcos[7][2] = {{0,1},{1,3},{2,3},{3,4},{4,5},{2,5},{4,1}};
   for (int i = 0; i < 7; ++i) GRAPHinsertArc( G, arcos[i][0], arcos[i][1]);

   vertex ord[maxV];  int ciclo[maxV];
   int k = GRAPHtopoCiclo( G, ord, ciclo);
   printf( "emitidos: %d de %d\n", k, G->V);
   printf( "esperado: vertices em ciclo = 1 3 4\nobtido:  ");
   for (vertex v = 0; v < G->V; ++v) if (ciclo[v]) printf( " %d", v);
   printf( "\n");
   return 0;
}

In [ ]:
compilar_e_rodar('ex2_ciclo.c')

## 5. A outra via: pós-ordem invertida da DFS

Kahn não é o único caminho. Rode a varredura DFS da aula anterior e observe o momento
em que a chamada `dfsR(v)` **termina**: nesse instante, todos os vértices alcançáveis a
partir de `v` já foram processados, logo `v` deve vir **antes** de todos eles na
ordenação.

Basta então registrar os vértices em pós-ordem e ler a lista de trás para frente. O
truque de implementação é escrever direto em `ord[--cnt2]`, preenchendo o vetor do fim
para o início: nenhuma pilha extra é necessária.

Compare a saída com a de Kahn — as duas ordens são **diferentes** e **ambas válidas**.
Um DAG pode ter muitas ordenações topológicas.

**Fonte:** Cormen et al. (2009), §22.4 — `TOPOLOGICAL-SORT` e teorema 22.12 (a pós-ordem invertida é uma ordenação topológica); Feofiloff (2020), cap. *Ordenação topológica*; Sedgewick e Wayne (2011), §4.2.

In [ ]:
%%writefile src/topo_dfs.c

/* ORDENACAO TOPOLOGICA - pos-ordem invertida da busca em profundidade. */
#include <stdio.h>
#include <stdlib.h>

#define vertex int
#define maxV 100

typedef struct node *link;
struct node { vertex w; link next; };
struct graph { int V; int A; link *adj; };
typedef struct graph *Graph;

static const char *nome[] = { "grafo.h", "grafo.c", "fila.c",
   "bfs.c", "topo.c", "main.c" };

static link NEWnode( vertex w, link next) {
   link a = malloc( sizeof (struct node));  a->w = w;  a->next = next;  return a;
}
Graph GRAPHinit( int V) {
   Graph G = malloc( sizeof *G);  G->V = V;  G->A = 0;
   G->adj = malloc( V * sizeof (link));
   for (vertex v = 0; v < V; ++v) G->adj[v] = NULL;
   return G;
}
void GRAPHinsertArc( Graph G, vertex v, vertex w) {
   G->adj[v] = NEWnode( w, G->adj[v]);  ++G->A;
}

static int pre[maxV];
static int cnt2;        /* ultima vaga livre de ord[], do fim ao inicio */
static int post[maxV], npost;   /* so para exibir a pos-ordem na tela */

static void dfsRtopo( Graph G, vertex v, vertex ord[]) {
   pre[v] = 0;                                    /* marca v */
   for (link a = G->adj[v]; a != NULL; a = a->next)
      if (pre[a->w] == -1) dfsRtopo( G, a->w, ord);
   post[npost++] = v;
   ord[--cnt2] = v;            /* POS-ordem: v ocupa a ultima vaga */
}

/* Supoe que G e um DAG. */
void GRAPHtopoDfs( Graph G, vertex ord[]) {
   for (vertex v = 0; v < G->V; ++v) pre[v] = -1;
   cnt2 = G->V;  npost = 0;
   for (vertex v = 0; v < G->V; ++v)
      if (pre[v] == -1) dfsRtopo( G, v, ord);
}

int main( void) {
   Graph G = GRAPHinit( 6);
   int arcos[6][2] = {{0,1},{1,3},{2,3},{3,4},{4,5},{2,5}};
   for (int i = 0; i < 6; ++i) GRAPHinsertArc( G, arcos[i][0], arcos[i][1]);

   vertex ord[maxV];
   GRAPHtopoDfs( G, ord);

   printf( "pos-ordem (ordem de TERMINO das chamadas):");
   for (int k = 0; k < npost; ++k) printf( " %d", post[k]);
   printf( "\nordem topologica (pos-ordem invertida):");
   for (int k = 0; k < G->V; ++k) printf( " %d", ord[k]);
   printf( "\n\n");
   for (int k = 0; k < G->V; ++k)
      printf( "  %d. %d (%s)\n", k + 1, ord[k], nome[ord[k]]);
   return 0;
}

In [ ]:
compilar_e_rodar('topo_dfs.c')
# Saida esperada: pos-ordem 5 4 3 1 0 2; ordem topologica 2 0 1 3 4 5.
# Kahn havia produzido 0 2 1 3 4 5 -- as duas respeitam todos os arcos.

## Desafio Final — em quantas rodadas dá para compilar tudo?

Com `make -j`, **quantos módulos quiser** podem ser compilados ao mesmo tempo, desde que
todas as suas dependências já tenham ficado prontas em rodadas anteriores. Qual é o
número **mínimo de rodadas** para construir todos os módulos do DAG?

Esse número é o comprimento do **maior caminho** do DAG (em vértices) — e ordenação
topológica é exatamente o que torna o cálculo fácil: percorra os vértices em ordem
topológica mantendo

$$\texttt{rod}[w] \;=\; \max\big(\texttt{rod}[w],\; \texttt{rod}[v] + 1\big)
\quad \text{para cada arco } v \to w,$$

começando com `rod[v] = 1` para todos. Quando $w$ é processado, todos os seus
predecessores já foram — é por isso que um único passe basta, sem recursão e sem
recalcular nada.

A resposta para o DAG dos seis módulos é **5**. Ao terminar, responda também:
quais módulos poderiam ser compilados na mesma rodada?

**Fonte:** Cormen et al. (2009), §24.2 (caminhos em DAG resolvidos em um único passe na ordem topológica) e §22.4; Levitin (2012), §4.2.

In [ ]:
%%writefile src/desafio_rodadas.c

#include <stdio.h>
#include <stdlib.h>

#define vertex int
#define maxV 100

typedef struct node *link;
struct node { vertex w; link next; };
struct graph { int V; int A; link *adj; };
typedef struct graph *Graph;

static link NEWnode( vertex w, link next) {
   link a = malloc( sizeof (struct node));  a->w = w;  a->next = next;  return a;
}
Graph GRAPHinit( int V) {
   Graph G = malloc( sizeof *G);  G->V = V;  G->A = 0;
   G->adj = malloc( V * sizeof (link));
   for (vertex v = 0; v < V; ++v) G->adj[v] = NULL;
   return G;
}
void GRAPHinsertArc( Graph G, vertex v, vertex w) {
   G->adj[v] = NEWnode( w, G->adj[v]);  ++G->A;
}

static vertex *fila;
static int cap, ini, fim, qtd;
void QUEUEinit( int N)   { cap = N;  fila = malloc( cap * sizeof (vertex));
                           ini = fim = qtd = 0; }
int  QUEUEempty( void)   { return qtd == 0; }
void QUEUEput( vertex v) { fila[fim] = v;  fim = (fim + 1) % cap;  ++qtd; }
vertex QUEUEget( void)   { vertex v = fila[ini];  ini = (ini + 1) % cap;
                           --qtd;  return v; }

/* DESAFIO: quantas rodadas de compilacao paralela, no minimo, para tudo?
   Nao ha limite de modulos por rodada: um modulo pode ser compilado
   assim que TODAS as suas dependencias estiverem prontas.
   Sugestao: percorra os vertices em ordem topologica (Kahn) mantendo
   rod[w] = max(rod[w], rod[v] + 1) para cada arco v -> w. */
int rodadasMinimas( Graph G) {
   /* TODO: implemente aqui e devolva o maior rod[v] */
   return 0;
}

int main( void) {
   Graph G = GRAPHinit( 6);
   int pre[6][2] = {{0,1},{1,3},{2,3},{3,4},{4,5},{2,5}};
   for (int i = 0; i < 6; ++i) GRAPHinsertArc( G, pre[i][0], pre[i][1]);
   printf( "esperado: 5\nobtido:   %d\n", rodadasMinimas( G));
   return 0;
}

In [ ]:
compilar_e_rodar('desafio_rodadas.c')

## Referências

- P. Feofiloff. *Algoritmos para Grafos (em linguagem C)*, IME-USP —
  capítulos *Busca em largura* e *Ordenação topológica*.
  <https://www.ime.usp.br/~pf/algoritmos_para_grafos/>
- T. H. Cormen, C. E. Leiserson, R. L. Rivest, C. Stein. *Introduction to Algorithms*,
  3ª ed. — §22.2 (BFS, lema 22.3 e teorema 22.5) e §22.4 (ordenação topológica).
- A. B. Kahn. *Topological sorting of large networks*. Communications of the ACM,
  5(11):558–562, 1962. <https://doi.org/10.1145/368996.369025>
- R. Sedgewick, K. Wayne. *Algorithms*, 4ª ed. — §4.1 (BFS) e §4.2 (DAGs).

A lista completa está em `../referencias.bib`.